# KeyDNA — ML Analysis Notebook
Exploratory data analysis and model evaluation for keystroke dynamics authentication

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sqlite3
import json
import sys
sys.path.insert(0, '../backend')

from ml.feature_extractor import extract_features, features_to_vector
from ml.authenticator import KeystrokeAuthenticator
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc

plt.style.use('dark_background')
sns.set_palette('husl')
print('✅ Imports OK')

In [ ]:
# Load data from database
conn = sqlite3.connect('../backend/keydna.db')
conn.row_factory = sqlite3.Row

# Get all training samples
samples = conn.execute("""
    SELECT ts.user_id, u.username, ts.features, ts.session_type
    FROM typing_samples ts
    JOIN users u ON u.id = ts.user_id
    WHERE ts.session_type = 'training'
""").fetchall()

print(f'Loaded {len(samples)} training samples from {len(set(s["username"] for s in samples))} users')

In [ ]:
# Build feature dataframe
records = []
for s in samples:
    features = json.loads(s['features'])
    features['user_id'] = s['user_id']
    features['username'] = s['username']
    records.append(features)

df = pd.DataFrame(records)
print(f'Feature matrix: {df.shape}')
df.describe()

In [ ]:
# ─── Visualize hold time distributions per user ───
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle('Keystroke Feature Distributions by User', fontsize=14, color='white')

features_to_plot = ['hold_mean', 'flight_mean', 'typing_speed']
titles = ['Hold Time (ms)', 'Flight Time (ms)', 'Typing Speed (c/s)']
colors = ['#3de8ff', '#22f0a4', '#a78bfa']

for ax, feat, title, color in zip(axes, features_to_plot, titles, colors):
    for user in df['username'].unique():
        user_df = df[df['username'] == user]
        ax.hist(user_df[feat].dropna(), bins=12, alpha=0.7, label=user)
    ax.set_title(title, color='white')
    ax.set_xlabel('Value', color='#8ba8be')
    ax.legend()

plt.tight_layout()
plt.savefig('feature_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ─── Feature Correlation Heatmap ───
numeric_cols = [c for c in df.columns if df[c].dtype in ['float64','int64'] and c not in ['user_id']]
corr = df[numeric_cols].corr()

fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0,
            linewidths=0.5, ax=ax, cbar_kws={'shrink': 0.8})
ax.set_title('Feature Correlation Matrix', fontsize=14, color='white', pad=20)
plt.tight_layout()
plt.savefig('feature_correlation.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ─── Model Evaluation (for a specific user) ───
from sklearn.model_selection import StratifiedKFold

user_id = 1  # Change this to the user you want to evaluate
user_samples = [json.loads(s['features']) for s in samples if s['user_id'] == user_id]

if len(user_samples) >= 5:
    auth = KeystrokeAuthenticator(user_id)
    feature_vectors = [features_to_vector(f) for f in user_samples]
    result = auth.train(feature_vectors)
    print('Training result:', result)
else:
    print(f'Not enough samples for user {user_id} (need 5, have {len(user_samples)})')

In [ ]:
# ─── ROC Curve Visualization ───
from sklearn.model_selection import cross_val_predict
from sklearn.preprocessing import label_binarize

# Simulated evaluation — replace with real cross-validation
np.random.seed(42)
n_genuine = 50
n_impostor = 50

# Simulate confidence scores
genuine_scores  = np.random.beta(5, 2, n_genuine)  # Skewed high
impostor_scores = np.random.beta(2, 5, n_impostor)  # Skewed low
all_scores = np.concatenate([genuine_scores, impostor_scores])
all_labels = np.array([1]*n_genuine + [0]*n_impostor)

fpr, tpr, _ = roc_curve(all_labels, all_scores)
roc_auc = auc(fpr, tpr)

fig, ax = plt.subplots(figsize=(8, 6))
ax.plot(fpr, tpr, color='#3de8ff', lw=2, label=f'ROC (AUC = {roc_auc:.3f})')
ax.plot([0,1], [0,1], color='#4a6278', lw=1, linestyle='--', label='Random Classifier')
ax.fill_between(fpr, tpr, alpha=0.1, color='#3de8ff')
ax.set_xlim([0, 1]); ax.set_ylim([0, 1.05])
ax.set_xlabel('False Positive Rate', color='white')
ax.set_ylabel('True Positive Rate', color='white')
ax.set_title('ROC Curve — KeyDNA Authentication', color='white', fontsize=13)
ax.legend(loc='lower right')
ax.grid(alpha=0.2)
plt.tight_layout()
plt.savefig('roc_curve.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'AUC-ROC: {roc_auc:.4f}')